# Machine Learning Foundations · Module 02
## From Predictions to Probabilities
**TeachReach · University of Alabama**

**Running example:** Campus Study Coach. In Module 1, Ava studied 2 hours and completed 3 practice sets. A toy linear model returned the score 55.

In this module we ask what the number 55 does **not** tell us: what outcomes remain possible, how evidence changes probabilities, and how to fit probabilistic models.

**Learning goals:** represent events and random variables; work with PMFs, PDFs, CDFs; compute expectation and variance; distinguish joint, marginal, and conditional probability; apply Bayes' rule; select probability distributions; maximize likelihood; build conditional probabilistic predictions.

All outcomes and parameters are **illustrative teaching examples**, not measurements from a real student population. This notebook runs in a free Colab Python runtime with PyTorch, NumPy and Matplotlib.

In [ ]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(27)
torch.set_printoptions(precision=4, sci_mode=False)
print("PyTorch:", torch.__version__)

## 0. Refresher: Module 1 in four operations
One student is a feature vector, a class is a matrix, and a linear model combines measurements and parameters using tensor operations.

- **x = [2,3]ᵀ**: Ava's two observed measurements.
- **w = [10,5]ᵀ, b = 20**: illustrative model parameters.
- **ŷ = xᵀw + b = 55**: point prediction.
- **ŷ = Xw + b**: vectorized predictions for multiple students.

**Checkpoint:** Which values were observed, which were selected as model parameters, and which value was computed?

In [ ]:
x = torch.tensor([2.0, 3.0])  # hours, practice sets
w = torch.tensor([10.0, 5.0]) # illustrative weights
b = torch.tensor(20.0)
mu_ava = x @ w + b
X = torch.tensor([[2.,3.],[4.,1.],[5.,2.]])
print("Ava:", mu_ava.item())
print("Class predictions:", X @ w + b)
assert mu_ava.item() == 55.0

## 1. Why introduce probability?
The same observed study measurements need not produce the same actual exam outcome. We distinguish the model's point prediction **ŷ = 55** from the random variable **Y**, representing the possible actual score.

A **sample space** Ω lists possible outcomes. An **event** A ⊆ Ω groups outcomes of interest, such as **A = {Y ≥ 60}**, where passing is defined as at least 60.

For whole-number scores, a toy sample space is {0,1,…,100}. For continuous approximations, use a real-valued distribution.

**Key distinction:** observed fractions estimate empirical frequencies; probability models encode assumptions that must be evaluated.

In [ ]:
# A simulated sample from an explicitly assumed model.
assumed_mu = torch.tensor(55.0)
assumed_std = torch.tensor(10.0)
score_model = torch.distributions.Normal(assumed_mu, assumed_std)
possible_scores = score_model.sample((12,))
print("Simulated outcomes:", possible_scores.round())
print("Event score >= 60:", possible_scores >= 60)
print("Fraction >= 60:", (possible_scores >= 60).float().mean().item())

### Quick check: empirical frequency versus probability
If 18 of 60 students in a sample passed, the observed passing fraction is (18/60=0.30). That fraction is a statistic; it need not be the exact probability for future students.

In [ ]:
observed_passes, observed_total = 18, 60
print("Empirical frequency:", observed_passes / observed_total)  # 0.3

## 2. Distributions: PMF, PDF, CDF
A **probability mass function** (PMF) gives P(Y=y) for discrete variables. These probabilities sum to 1.

A **probability density function** (PDF), f(y), describes continuous variables. Density heights are not point probabilities: for continuous Y, **P(Y=y)=0**, and interval probabilities come from areas.

The **cumulative distribution function** (CDF) is **F(t)=P(Y≤t)**. Thus **P(Y>t)=1−F(t)**.

We use Normal(55, 10²) as an illustration. Exam scores are bounded, so this distribution need not fit an actual grading process.

In [ ]:
pass_flag = torch.distributions.Bernoulli(probs=torch.tensor(0.70))
print("Bernoulli P(0), P(1):", pass_flag.log_prob(torch.tensor([0.,1.])).exp())

normal = torch.distributions.Normal(loc=torch.tensor(55.0), scale=torch.tensor(10.0))
t = torch.tensor(60.0)
print("Density at 60:", normal.log_prob(t).exp().item())
print("P(Y > 60):", (1-normal.cdf(t)).item())  # approximately 0.3085
print("P(50 <= Y <= 60):",
      (normal.cdf(torch.tensor(60.0)) - normal.cdf(torch.tensor(50.0))).item())

In [ ]:
grid = torch.linspace(20, 90, 250)
density = normal.log_prob(grid).exp()
plt.figure(figsize=(8, 3.8))
plt.plot(grid.numpy(), density.numpy(), linewidth=2, label="Normal(55, 10²)")
plt.fill_between(grid.numpy(), 0, density.numpy(), where=(grid.numpy() >= 60), alpha=.25)
plt.axvline(60, linestyle="--", label="Threshold 60")
plt.xlabel("Score"); plt.ylabel("Probability density")
plt.title("Probability above a threshold is an area, not a curve height")
plt.legend(); plt.show()

## 3. Expectation, variance, standard deviation
Expectation is the probability-weighted mean:

**E[Y] = Σᵧ y·P(Y=y)** for discrete Y, or the integral of y·f(y) over the real line for continuous Y.

Variance measures squared spread: **Var(Y) = E[(Y−μ)²]**, where μ=E[Y].
The standard deviation **σ = √Var(Y)** has the original measurement units.

In [ ]:
outcomes = torch.tensor([40., 60.])
probabilities = torch.tensor([0.5, 0.5])
mean = (outcomes*probabilities).sum()
variance = ((outcomes-mean)**2 * probabilities).sum()
print("E[Y] =",mean.item(),"Var(Y) =",variance.item(),"Std =",variance.sqrt().item())
assert mean.item()==50.0 and variance.item()==100.0

### Same mean, different spread
A Normal distribution with **σ = 5** is narrower than one with **σ = 15**, even when both have **μ = 55**. Observe how the probability of passing above 60 changes with the spread.

In [ ]:
for std in [5., 10., 15.]:
    d = torch.distributions.Normal(55., std)
    print(f"mean=55, std={std:>4}: P(Y>60)={float(1-d.cdf(torch.tensor(60.))):.4f}")

## 4. Sampling and the law of large numbers
Sampling generates possible outcomes under a chosen model. Under standard assumptions, the sample mean tends toward the distribution mean as the number of samples increases.

We will keep the model fixed and change only the number of simulated observations.

In [ ]:
torch.manual_seed(27)
draws = normal.sample((10000,))
for n in [5, 20, 100, 1000, 10000]:
    print(f"n={n:5}: sample mean={draws[:n].mean().item():.3f}")
plt.figure(figsize=(7.8,3.5))
running_mean = draws.cumsum(0) / torch.arange(1,10001)
plt.plot(range(1,10001), running_mean.numpy())
plt.axhline(55, color="black", linestyle="--", label="Model mean 55")
plt.xscale("log");plt.xlabel("Number of draws (log scale)")
plt.ylabel("Running sample mean");plt.legend();plt.show()

## 5. Joint, marginal and conditional probability
In an **illustrative** class of 100 students:

| Practice sets | Passed | Did not pass | Total |
|---|---:|---:|---:|
| 3 or more | 36 | 9 | 45 |
| Fewer than 3 | 22 | 33 | 55 |
| Total | 58 | 42 | 100 |

Different probability questions require different denominators.

- **Joint:** P(Pass ∩ Practice)=36/100.
- **Marginal:** P(Pass)=58/100.
- **Conditional:** P(Pass | Practice)=36/45.
- **Reverse conditional:** P(Practice | Pass)=36/58.

Observed association does not establish causation.

In [ ]:
counts = torch.tensor([[36.,9.],[22.,33.]])
n = counts.sum()
p_joint = counts[0,0]/n
p_pass = counts[:,0].sum()/n
p_pass_given_practice = counts[0,0]/counts[0,:].sum()
p_practice_given_pass = counts[0,0]/counts[:,0].sum()
print({k:round(v.item(),4) for k,v in {
    "joint":p_joint,"marginal_pass":p_pass,
    "pass_given_practice":p_pass_given_practice,
    "practice_given_pass":p_practice_given_pass
}.items()})
assert abs(p_pass_given_practice.item()-.8)<1e-7

### Independence
Two events A and B are independent if **P(A | B)=P(A)** (when P(B)>0), equivalently **P(A ∩ B)=P(A)P(B)**.

Our illustrative table does not satisfy independence: 0.80 ≠ 0.58. This statement describes the table, not a causal intervention.

In [ ]:
p_practice = counts[0,:].sum()/n
print("Joint:", p_joint.item())
print("Product of marginals:", (p_pass*p_practice).item())
print("Independent in toy table?", torch.isclose(p_joint,p_pass*p_practice).item())

## 6. Bayes' rule: revise probabilities after evidence
Suppose before a practice test **P(Pass)=0.60**. A strong test result has:

- **P(Strong | Pass)=0.80** (sensitivity).
- **P(Strong | Fail)=0.25** (false-positive rate).

Bayes' rule updates the prior by multiplying it by the likelihood and dividing by the total evidence probability.

**P(Pass | Strong) = [0.80 × 0.60] / [0.80 × 0.60 + 0.25 × 0.40] = 0.48/0.58 ≈ 0.828.**

The denominator counts evidence under both passing and failing hypotheses.

In [ ]:
prior = torch.tensor([0.60, 0.40])   # pass, fail
likelihood_strong = torch.tensor([0.80, 0.25])
weights = prior * likelihood_strong
evidence_probability = weights.sum()
posterior_strong = weights / evidence_probability
print("P(strong):",evidence_probability.item())
print("Posterior [pass,fail]:",posterior_strong)
assert abs(posterior_strong[0].item() - 0.48/0.58) < 1e-6

### Weak evidence is a different update
A weak result is the complement of strong evidence: **P(Weak | Pass)=0.20** and **P(Weak | Fail)=0.75**.

The posterior probability of passing becomes **0.12/(0.12+0.30)≈0.286**.

In [ ]:
likelihood_weak = 1 - likelihood_strong
posterior_weak = (prior*likelihood_weak)
posterior_weak /= posterior_weak.sum()
print("P(Pass | weak):",posterior_weak[0].item())

## 7. Choose a distribution that matches the question
- **Bernoulli:** one binary event (pass or not).
- **Binomial:** count of successful binary trials under a fixed-trial, common-(p), independent-trial model.
- **Categorical:** one label among a finite set of classes.
- **Normal:** continuous numerical outcomes when a bell-shaped model is a reasonable approximation.

This choice is about the meaning of the outcome, not which distribution is easiest to code.

In [ ]:
bern = torch.distributions.Bernoulli(probs=torch.tensor(.8))
count_pass = torch.distributions.Binomial(total_count=10,probs=torch.tensor(.8))
band = torch.distributions.Categorical(probs=torch.tensor([.2,.5,.3]))
score = torch.distributions.Normal(loc=torch.tensor(55.),scale=torch.tensor(10.))
print("Single pass:",bern.sample().item())
print("Passes among 10:",count_pass.sample().item())
print("Category ID:",band.sample().item())
print("Possible score:",score.sample().item())

## 8. Likelihood and log-likelihood
Given fixed observed values y₁,…,yₙ and a distribution with parameters θ, likelihood is a **function of θ**:

- **L(θ) = ∏ᵢ p(yᵢ | θ)**.
- **ℓ(θ) = log L(θ) = Σᵢ log p(yᵢ | θ)**.

Maximizing log-likelihood is equivalent to maximizing likelihood. Minimizing **negative log-likelihood** is equivalent to maximizing log-likelihood.

For continuous variables, p(yᵢ|θ) is a density value, not the probability of the exact observation.

In [ ]:
observed = torch.tensor([42.,48.,55.,61.,69.])
candidate = torch.distributions.Normal(loc=torch.tensor(55.),scale=torch.tensor(10.))
log_likelihood = candidate.log_prob(observed).sum()
print("Log likelihood:",log_likelihood.item())
print("Negative log likelihood:",(-log_likelihood).item())

## 9. Gaussian maximum-likelihood estimates
For independent observations assumed to share a Normal distribution, the MLEs are:

- **μ̂ = (1/n)Σᵢyᵢ**
- **σ̂² = (1/n)Σᵢ(yᵢ−μ̂)²**

With example scores [42,48,55,61,69], the estimates are **μ̂=55**, **σ̂²=90**, **σ̂≈9.487**.

The unbiased sample variance differs because it divides by n−1.

In [ ]:
mu_mle = observed.mean()
var_mle = ((observed-mu_mle)**2).mean()
sigma_mle = torch.sqrt(var_mle)
print("Mean MLE:",mu_mle.item(),"Variance MLE:",var_mle.item(),"Sigma:",sigma_mle.item())
print("Unbiased variance:",observed.var(unbiased=True).item())
assert abs(mu_mle.item()-55)<1e-6 and abs(var_mle.item()-90)<1e-6

### Fit the Normal model with gradient descent
This uses the same pattern as Module 1: parameters are tensors, we define a loss, compute gradients, and update parameters.

We optimize **μ** and **log σ**, which ensures that **σ = exp(log σ) > 0**. The analytic MLE from the previous cell is a reference for checking the result.

In [ ]:
fit_mu = torch.nn.Parameter(torch.tensor(50.))
fit_log_sigma = torch.nn.Parameter(torch.log(torch.tensor(12.)))
optimizer = torch.optim.Adam([fit_mu, fit_log_sigma],lr=.08)
history=[]
for step in range(450):
    optimizer.zero_grad()
    dist = torch.distributions.Normal(fit_mu, fit_log_sigma.exp())
    nll = -dist.log_prob(observed).sum()
    nll.backward()
    optimizer.step()
    history.append(nll.item())
print("Learned mu:",fit_mu.item(),"sigma:",fit_log_sigma.exp().item())
print("Analytic mu:",mu_mle.item(),"sigma:",sigma_mle.item())
plt.figure(figsize=(7.8,3.3))
plt.plot(history)
plt.xlabel("Optimization step");plt.ylabel("Negative log-likelihood")
plt.title("Fitting distribution parameters by gradient descent")
plt.show()

## 10. Probabilistic regression
The tensor model from Module 1 can produce a **conditional mean**:

- **μθ(x)=wᵀx+b**
- **Y | X=x ∼ Normal(μθ(x), σ²)**

Here we set σ=10 to illustrate the mechanics. A fitted model should estimate and validate the spread, rather than choose it without evidence.

For Ava, the illustrative **P(Y>60 | x)** is about 0.309.

In [ ]:
mu = x @ w + b
conditional = torch.distributions.Normal(mu, torch.tensor(10.))
p_pass = 1-conditional.cdf(torch.tensor(60.))
print("Point prediction:",mu.item())
print("P(score > 60 | Ava's features):",p_pass.item())

## 11. Probabilistic classification
If the outcome is binary (pass/fail), the logistic link maps an unconstrained score to a probability:

**pθ(x)=sigmoid(wᵀx+b)=1/[1+exp(−(wᵀx+b))]**.

A Bernoulli model describes the binary label. We train a tiny demonstration model with **binary cross-entropy with logits**, a numerically stable negative log-likelihood for Bernoulli outcomes.

In [ ]:
# Small illustrative dataset; not educational research data.
X_train = torch.tensor([
 [1.,0.],[1.,2.],[2.,1.],[3.,1.],[4.,3.],[5.,2.],
 [2.,0.],[0.,2.],[3.,3.],[4.,1.]
])
y_train = torch.tensor([0.,0.,0.,1.,1.,1.,0.,0.,1.,1.])
weights = torch.nn.Parameter(torch.zeros(2))
bias = torch.nn.Parameter(torch.tensor(0.0))
opt = torch.optim.Adam([weights,bias],lr=.08)
for step in range(350):
    opt.zero_grad()
    logits = X_train @ weights + bias
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits,y_train)
    loss.backward()
    opt.step()
with torch.no_grad():
    predictions = torch.sigmoid(X_train@weights+bias)
print("Predicted probabilities:",predictions.round(decimals=3))
print("Training NLL per example:",loss.item())

## 12. Check the quality of probabilities
Predictive performance should be measured using **held-out data**. Negative log-likelihood penalizes assigning low probability to observed outcomes. Calibration compares predicted probabilities with observed event frequencies across appropriate groups.

The following is a simulation of calibration on synthetic data, not an evaluation of the Campus Study Coach. A real study would need a careful train/validation/test split and appropriate uncertainty analysis.

In [ ]:
torch.manual_seed(42)
true_probs = torch.linspace(.05,.95,1000)
outcomes = torch.bernoulli(true_probs)
bins = torch.linspace(0,1,11)
bin_conf, bin_freq = [], []
for i in range(10):
    mask = (true_probs >= bins[i]) & (true_probs < bins[i+1] if i<9 else true_probs<=1.)
    if mask.any():
        bin_conf.append(true_probs[mask].mean().item())
        bin_freq.append(outcomes[mask].mean().item())
plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1],"--",label="Perfect calibration")
plt.plot(bin_conf,bin_freq,"o-",label="Simulated probabilities")
plt.xlabel("Mean predicted probability");plt.ylabel("Observed fraction")
plt.title("Illustrative calibration curve");plt.legend();plt.show()

## 13. Knowledge check (solve before opening the answers)
1. A cohort has 45 students with ≥3 practice sets; 36 pass. Compute P(Pass | Practice≥3).
2. For Bernoulli P(Y=1)=0.7, compute P(Y=0).
3. Explain why Normal density at 60 is not P(Y=60).
4. For the five scores [42,48,55,61,69], compute Gaussian MLE variance.
5. Compute P(Pass | Strong) with prior 0.6, sensitivity 0.8 and false-positive rate 0.25.
6. Give one reason an assumed Normal model may be inappropriate for bounded exam scores.

In [ ]:
# Optional self-checks. Fill in your answers, then run.
student_conditional = None
student_bernoulli_zero = None
student_mle_variance = None
student_bayes = None
# Uncomment after filling in the values:
# assert abs(student_conditional - 0.80) < 1e-6
# assert abs(student_bernoulli_zero - 0.30) < 1e-6
# assert abs(student_mle_variance - 90) < 1e-6
# assert abs(student_bayes - 0.48/0.58) < 1e-3

### Worked answers
1. **36/45=0.80.** Use only the conditioning group as denominator.
2. **1−0.7=0.30.**
3. A PDF is a density; exact points have probability zero in a continuous model, while intervals have positive probability.
4. Deviations from 55 are [−13,−7,0,6,14]; squared deviations sum to 450. MLE variance is **450/5=90**.
5. **0.48/(0.48+0.10)=0.827586…**
6. Normal permits values below 0 and above 100 while exam percentages are bounded.

**Bridge to Module 03.** Keep Ava’s predicted mean of 55 and assumed standard deviation of 10. Suppose her observed exam score is 60. The Gaussian negative log-likelihood scores that observation. How much should we change each weight and the bias to lower the loss?